# 02. CTR Data Preprocessing

Process the large Parquet files in batches. Split rows reproducibly into train and validation sets, estimate numeric imputation values from training rows only, and retain test IDs for submission. The default output directory is `data/processed/` next to the raw data directory.

The transformed data can still require several gigabytes of disk space. Existing outputs are never removed unless `OVERWRITE_PROCESSED` is explicitly set to `True`.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

def find_data_dir():
    configured_dir = os.environ.get('CTR_DATA_DIR')
    if configured_dir:
        candidate = Path(configured_dir).expanduser().resolve()
        if (candidate / 'train.parquet').is_file() and (candidate / 'test.parquet').is_file():
            return candidate
        raise FileNotFoundError(f'CTR_DATA_DIR does not contain train.parquet and test.parquet: {candidate}')
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates = [base / 'data' / 'raw', base / 'data_project' / 'data' / 'raw']
        for candidate in candidates:
            if (candidate / 'train.parquet').is_file() and (candidate / 'test.parquet').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Could not find data/raw/train.parquet and test.parquet.')

DATA_DIR = find_data_dir()
TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
OUTPUT_DIR = Path(os.environ.get('CTR_OUTPUT_DIR', DATA_DIR.parent / 'processed')).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = 'clicked'
CAT_COLUMNS = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour', 'seq']
train_file = pq.ParquetFile(TRAIN_PATH)
test_file = pq.ParquetFile(TEST_PATH)
FEATURE_COLUMNS = [column for column in train_file.schema.names if column != TARGET]
NUM_COLUMNS = [column for column in FEATURE_COLUMNS if column not in CAT_COLUMNS]
MISSING_CATEGORY = '__MISSING__'
BATCH_SIZE = 100_000
VALIDATION_FRACTION = 0.10
RANDOM_SEED = 2025
OVERWRITE_PROCESSED = False

assert test_file.schema.names == ['ID'] + FEATURE_COLUMNS
print(f'Raw data: {DATA_DIR}')
print(f'Processed output: {OUTPUT_DIR}')
print(f'{len(FEATURE_COLUMNS)} features: {len(NUM_COLUMNS)} numeric and {len(CAT_COLUMNS)} categorical')

In [ ]:
output_paths = {
    'train': OUTPUT_DIR / 'train.parquet',
    'validation': OUTPUT_DIR / 'validation.parquet',
    'test': OUTPUT_DIR / 'test.parquet',
    'medians': OUTPUT_DIR / 'numeric_medians.json',
}
existing_outputs = [path for path in output_paths.values() if path.exists()]
if existing_outputs and not OVERWRITE_PROCESSED:
    raise FileExistsError(
        'Processed outputs already exist. Set OVERWRITE_PROCESSED = True to replace these files: '
        + ', '.join(str(path) for path in existing_outputs)
    )
if OVERWRITE_PROCESSED:
    for path in existing_outputs:
        path.unlink()

split_rng = np.random.default_rng(RANDOM_SEED)
median_samples = []
for batch in train_file.iter_batches(batch_size=BATCH_SIZE):
    frame = batch.to_pandas()
    validation_mask = split_rng.random(len(frame)) < VALIDATION_FRACTION
    train_part = frame.loc[~validation_mask, NUM_COLUMNS]
    sample_size = min(3_000, len(train_part))
    if sample_size:
        median_samples.append(
            train_part.sample(n=sample_size, random_state=RANDOM_SEED + len(median_samples))
        )

median_sample = pd.concat(median_samples, ignore_index=True)
numeric_medians = median_sample.median().fillna(0.0).to_dict()
print(f'Rows used to estimate numeric medians: {len(median_sample):,}')
display(pd.Series(numeric_medians, name='training_sample_median').head(20))

In [ ]:
def prepare_features(frame, include_target=False, include_id=False):
    output_columns = []
    if include_id:
        frame['ID'] = frame['ID'].astype('string')
        output_columns.append('ID')
    for column in CAT_COLUMNS:
        frame[column] = frame[column].astype('string').fillna(MISSING_CATEGORY)
    for column in NUM_COLUMNS:
        frame[column] = frame[column].fillna(numeric_medians[column]).astype('float32')
    output_columns.extend(FEATURE_COLUMNS)
    if include_target:
        frame[TARGET] = frame[TARGET].astype('int8')
        output_columns.append(TARGET)
    return frame[output_columns]

writers = {'train': None, 'validation': None}
row_counts = {'train': 0, 'validation': 0}
target_counts = {'train': {}, 'validation': {}}
split_rng = np.random.default_rng(RANDOM_SEED)
try:
    for batch in train_file.iter_batches(batch_size=BATCH_SIZE):
        frame = batch.to_pandas()
        validation_mask = split_rng.random(len(frame)) < VALIDATION_FRACTION
        for split_name, selected in [('train', ~validation_mask), ('validation', validation_mask)]:
            part = prepare_features(frame.loc[selected].copy(), include_target=True)
            table = pa.Table.from_pandas(part, preserve_index=False)
            if writers[split_name] is None:
                writers[split_name] = pq.ParquetWriter(
                    output_paths[split_name], table.schema, compression='snappy'
                )
            writers[split_name].write_table(table)
            row_counts[split_name] += len(part)
            for label, count in part[TARGET].value_counts().to_dict().items():
                label = int(label)
                target_counts[split_name][label] = target_counts[split_name].get(label, 0) + int(count)
finally:
    for writer in writers.values():
        if writer is not None:
            writer.close()

print('Rows by split:', row_counts)
print('Target counts by split:', target_counts)

In [ ]:
test_writer = None
test_rows = 0
try:
    for batch in test_file.iter_batches(batch_size=BATCH_SIZE):
        frame = batch.to_pandas()
        part = prepare_features(frame, include_id=True)
        table = pa.Table.from_pandas(part, preserve_index=False)
        if test_writer is None:
            test_writer = pq.ParquetWriter(output_paths['test'], table.schema, compression='snappy')
        test_writer.write_table(table)
        test_rows += len(part)
finally:
    if test_writer is not None:
        test_writer.close()

with output_paths['medians'].open('w', encoding='utf-8') as file:
    json.dump(numeric_medians, file, indent=2)
print(f'Test rows: {test_rows:,}')
for split_name in ['train', 'validation', 'test']:
    path = output_paths[split_name]
    print(f'{path.name}: {path.stat().st_size / (1024 ** 3):.2f} GiB')

In [ ]:
processed_train = pq.ParquetFile(output_paths['train'])
processed_validation = pq.ParquetFile(output_paths['validation'])
processed_test = pq.ParquetFile(output_paths['test'])
assert processed_train.metadata.num_rows == row_counts['train']
assert processed_validation.metadata.num_rows == row_counts['validation']
assert processed_test.metadata.num_rows == test_rows
assert processed_train.schema.names == FEATURE_COLUMNS + [TARGET]
assert processed_validation.schema.names == FEATURE_COLUMNS + [TARGET]
assert processed_test.schema.names == ['ID'] + FEATURE_COLUMNS

train_check = next(processed_train.iter_batches(batch_size=2_000)).to_pandas()
validation_check = next(processed_validation.iter_batches(batch_size=2_000)).to_pandas()
test_check = next(processed_test.iter_batches(batch_size=2_000)).to_pandas()
assert not train_check.isna().any().any()
assert not validation_check.isna().any().any()
assert not test_check.isna().any().any()

print('Preprocessed outputs passed schema, row-count, and null checks.')
print(f"Train: {processed_train.metadata.num_rows:,}; validation: {processed_validation.metadata.num_rows:,}; test: {processed_test.metadata.num_rows:,}")

## Preprocessing decisions

- Missing categorical values are preserved as a dedicated `__MISSING__` category.
- Missing numeric values are filled using medians sampled from training rows only; validation and test rows do not contribute to the imputation values.
- The fixed-seed split is performed by row. The target is excluded from features, and the test `ID` is retained only for prediction output.
- Categorical encoding and numeric scaling are left to the selected model pipeline. This avoids imposing an encoding on models that can handle categorical strings directly.